## Natural Language Processing with Naive Bayes with Pyspark

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('nlp_nb').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/21 17:08:46 WARN Utils: Your hostname, aditya-HP-Laptop-15s-eq1xxx, resolves to a loopback address: 127.0.1.1; using 10.103.210.123 instead (on interface wlo1)
26/04/21 17:08:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 17:08:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
!head SMSSpamCollection

ham	Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
ham	Ok lar... Joking wif u oni...
spam	Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
ham	U dun say so early hor... U c already then say...
ham	Nah I don't think he goes to usf, he lives around here though
spam	FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, £1.50 to rcv
ham	Even my brother is not like to speak with me. They treat me like aids patent.
ham	As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune
spam	WINNER!! As a valued network customer you have been selected to receivea £900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.
spam	H

In [4]:
df = spark.read.csv('SMSSpamCollection', inferSchema=True, sep='\t')

In [6]:
df.show()

+----+--------------------+
| _c0|                 _c1|
+----+--------------------+
| ham|Go until jurong p...|
| ham|Ok lar... Joking ...|
|spam|Free entry in 2 a...|
| ham|U dun say so earl...|
| ham|Nah I don't think...|
|spam|FreeMsg Hey there...|
| ham|Even my brother i...|
| ham|As per your reque...|
|spam|WINNER!! As a val...|
|spam|Had your mobile 1...|
| ham|I'm gonna be home...|
|spam|SIX chances to wi...|
|spam|URGENT! You have ...|
| ham|I've been searchi...|
| ham|I HAVE A DATE ON ...|
|spam|XXXMobileMovieClu...|
| ham|Oh k...i'm watchi...|
| ham|Eh u remember how...|
| ham|Fine if thats th...|
|spam|England v Macedon...|
+----+--------------------+
only showing top 20 rows


In [7]:
df = df.withColumnRenamed("_c0", "class").withColumnRenamed("_c1", "text")

#### Clean Data

In [8]:
from pyspark.sql.functions import length

In [9]:
df = df.withColumn("length", length(df['text']))

In [10]:
df.show()

+-----+--------------------+------+
|class|                text|length|
+-----+--------------------+------+
|  ham|Go until jurong p...|   111|
|  ham|Ok lar... Joking ...|    29|
| spam|Free entry in 2 a...|   155|
|  ham|U dun say so earl...|    49|
|  ham|Nah I don't think...|    61|
| spam|FreeMsg Hey there...|   147|
|  ham|Even my brother i...|    77|
|  ham|As per your reque...|   160|
| spam|WINNER!! As a val...|   157|
| spam|Had your mobile 1...|   154|
|  ham|I'm gonna be home...|   109|
| spam|SIX chances to wi...|   136|
| spam|URGENT! You have ...|   155|
|  ham|I've been searchi...|   196|
|  ham|I HAVE A DATE ON ...|    35|
| spam|XXXMobileMovieClu...|   149|
|  ham|Oh k...i'm watchi...|    26|
|  ham|Eh u remember how...|    81|
|  ham|Fine if thats th...|    56|
| spam|England v Macedon...|   155|
+-----+--------------------+------+
only showing top 20 rows


In [11]:
df.groupby('class').mean().show()

+-----+-----------------+
|class|      avg(length)|
+-----+-----------------+
|  ham|71.45431945307645|
| spam|138.6706827309237|
+-----+-----------------+



### Feature Transformation

#### Creating Assemblers and Features

In [14]:
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, StringIndexer, IDF

In [15]:
tokenizer = Tokenizer(inputCol='text', outputCol='token_text')
stop_word_remover = StopWordsRemover(inputCol='token_text', outputCol='stop_tokens')
count_vec = CountVectorizer(inputCol='stop_tokens', outputCol='c_vec')
idf = IDF(inputCol='c_vec', outputCol='tf_idf')
ham_spam_to_num = StringIndexer(inputCol='class', outputCol='label')

In [16]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.linalg import Vector

In [18]:
cleaned = VectorAssembler(inputCols=['tf_idf', 'length'], outputCol='features')

### Naive Bayes Model

In [19]:
from pyspark.ml.classification import NaiveBayes

In [20]:
nb = NaiveBayes()

#### Pipeline

In [21]:
from pyspark.ml import Pipeline

In [22]:
pipeline = Pipeline(stages=[
    ham_spam_to_num,
    tokenizer,
    stop_word_remover,
    count_vec,
    idf,
    cleaned
])

In [23]:
cleaner = pipeline.fit(df)

In [24]:
clean_df = cleaner.transform(df)

In [25]:
clean_df.show()

+-----+--------------------+------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|class|                text|length|label|          token_text|         stop_tokens|               c_vec|              tf_idf|            features|
+-----+--------------------+------+-----+--------------------+--------------------+--------------------+--------------------+--------------------+
|  ham|Go until jurong p...|   111|  0.0|[go, until, juron...|[go, jurong, poin...|(13423,[7,11,31,6...|(13423,[7,11,31,6...|(13424,[7,11,31,6...|
|  ham|Ok lar... Joking ...|    29|  0.0|[ok, lar..., joki...|[ok, lar..., joki...|(13423,[0,24,301,...|(13423,[0,24,301,...|(13424,[0,24,301,...|
| spam|Free entry in 2 a...|   155|  1.0|[free, entry, in,...|[free, entry, 2, ...|(13423,[2,13,19,3...|(13423,[2,13,19,3...|(13424,[2,13,19,3...|
|  ham|U dun say so earl...|    49|  0.0|[u, dun, say, so,...|[u, dun, say, ear...|(13423,[0,70,80,1...|(13423,[0,70,8

### Train Model and Evaluation

In [26]:
clean_df = clean_df.select(['label', 'features'])

In [27]:
(train, test) = clean_df.randomSplit([0.7, 0.3], seed=42)

In [28]:
pred = nb.fit(train)

26/04/21 17:30:59 WARN DAGScheduler: Broadcasting large task binary with size 1027.9 KiB
26/04/21 17:31:01 WARN DAGScheduler: Broadcasting large task binary with size 1023.0 KiB
                                                                                

In [29]:
res = pred.transform(test)

In [30]:
res.show()

26/04/21 17:31:25 WARN DAGScheduler: Broadcasting large task binary with size 1242.6 KiB
[Stage 23:>                                                         (0 + 1) / 1]

+-----+--------------------+--------------------+--------------------+----------+
|label|            features|       rawPrediction|         probability|prediction|
+-----+--------------------+--------------------+--------------------+----------+
|  0.0|(13424,[0,1,2,41,...|[-1066.4444494620...|[1.0,2.9801047802...|       0.0|
|  0.0|(13424,[0,1,5,20,...|[-808.28928301010...|[1.0,4.7601546641...|       0.0|
|  0.0|(13424,[0,1,7,8,1...|[-1167.3054457630...|[1.0,2.6335649563...|       0.0|
|  0.0|(13424,[0,1,7,15,...|[-657.95991822322...|[1.0,2.6681187262...|       0.0|
|  0.0|(13424,[0,1,12,33...|[-444.48015999387...|[1.0,1.8748881516...|       0.0|
|  0.0|(13424,[0,1,14,18...|[-1362.7802170498...|[1.0,1.2529954030...|       0.0|
|  0.0|(13424,[0,1,14,31...|[-215.54775472262...|[1.0,4.8844219551...|       0.0|
|  0.0|(13424,[0,1,18,20...|[-830.46884103326...|[1.0,8.0551837369...|       0.0|
|  0.0|(13424,[0,1,21,27...|[-774.83487436693...|[1.0,2.5889764398...|       0.0|
|  0.0|(13424,[0

In [31]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
evals = MulticlassClassificationEvaluator()
acc = evals.evaluate(res)
print(f"Accuracy: {acc * 100}%")